In [17]:
%reload_ext autoreload
%autoreload 2

import qiskit_metal as metal
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict, Headings
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround
from qiskit_metal.qlibrary.terminations.launchpad_wb_driven import LaunchpadWirebondDriven
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight
import pyEPR as epr
import numpy as np

In [18]:
design = designs.DesignPlanar({}, True)
design.chips.main.size['size_x'] = '5mm'
design.chips.main.size['size_y'] = '5mm'
design.variables['cpw_width'] = '10 um'
design.variables['cpw_gap'] = '6 um'
gui = MetalGUI(design)

In [19]:
type(design)

qiskit_metal.designs.design_planar.DesignPlanar

In [20]:
from qiskit_metal.qlibrary.qubits.transmon_cross import TransmonCross

design.delete_all_components()

TransmonCross.get_template_options(design)

{'pos_x': '0.0um',
 'pos_y': '0.0um',
 'orientation': '0.0',
 'chip': 'main',
 'layer': '1',
 'connection_pads': {},
 '_default_connection_pads': {'connector_type': '0',
  'claw_length': '30um',
  'ground_spacing': '5um',
  'claw_width': '10um',
  'claw_gap': '6um',
  'connector_location': '0'},
 'cross_width': '20um',
 'cross_length': '200um',
 'cross_gap': '20um',
 'hfss_inductance': '10nH',
 'hfss_capacitance': 0,
 'hfss_resistance': 0,
 'hfss_mesh_kw_jj': 7e-06,
 'q3d_inductance': '10nH',
 'q3d_capacitance': 0,
 'q3d_resistance': 0,
 'q3d_mesh_kw_jj': 7e-06,
 'gds_cell_name': 'my_other_junction'}

In [21]:
pos_ro_x = 2500
cpw_width = 10
epsilon = 11.4
fillet='99.99um'
cpw_options = Dict(
    lead=Dict(
        start_straight='50um',
        end_straight='250um'
    ),
    fillet=fillet,
    meander=Dict(spacing="200um")
)


def pos_from_offset(offset):
    return pos_ro_x + offset

def quarter_wave_length(frequency_ghz, epsilon_eff):
    """input in GHz, output in um"""
    c = 3e8
    frequency_hz = frequency_ghz * 1e9
    wavelength = c / (frequency_hz * (epsilon_eff ** 0.5))
    quarter_wavelength = wavelength / 4
    return quarter_wavelength * 1e6

def generate_readout_frequencies(center_freq_ghz=7.4, spacing_mhz=30, num_qubits=6):
    start_freq = center_freq_ghz - (spacing_mhz * (num_qubits - 1) / 2) / 1000
    return [round(start_freq + i * spacing_mhz / 1000, 6) for i in range(num_qubits)]

def connect(cpw_name: str, pin1_comp_name: str, pin1_comp_pin: str, 
            pin2_comp_name: str, pin2_comp_pin: str,
            length: str, asymmetry='0 um'):
    """Connect two pins with a CPW."""
    myoptions = Dict(
        pin_inputs=Dict(
            start_pin=Dict(
                component=pin1_comp_name,
                pin=pin1_comp_pin),
            end_pin=Dict(
                component=pin2_comp_name,
                pin=pin2_comp_pin)),
        total_length=length)
    myoptions.update(cpw_options)
    myoptions.meander.asymmetry = asymmetry
    return RouteMeander(design, cpw_name, myoptions)

frequencies = generate_readout_frequencies()

length_um_list = [quarter_wave_length(freq, epsilon) for freq in frequencies]

## Build stuff

In [22]:
# qubit
xmon_options = dict(
    pos_x = '1.0mm',
    pos_y = '4.25mm',
    orientation = '270',
    cross_width = '80um',
    cross_length = '125um',
    cross_gap = '5um',
    connection_pads=dict(
        connector_1 = dict(connector_location = '90',
            connector_type = '0',
            ground_spacing = '5um', 
            claw_length = '30um',
            claw_width = '20um',
            claw_gap = '5um',
        )
    ),
)

Qubit_1 = TransmonCross(design, 'Qubit_1', options=xmon_options)
gui.rebuild()



In [ ]:
# resonator
offset_ro_x_1 = -25
pos_x_cp1 = pos_from_offset(offset_ro_x_1)
pos_y_cp1 = 4250


otg = OpenToGround(design, 'coupling_pin', options=dict(
            pos_x=f'{pos_x_cp1}um',
            pos_y=f'{pos_y_cp1}um',
            orientation="270",
            ))
RouteMeander(design, 'cpw1', Dict(
            total_length='4037um',
            hfss_wire_bonds = True,
            fillet='99.9um',
            lead=dict(start_straight='100um'),
            pin_inputs=Dict(
                start_pin=Dict(component='Qubit_1', pin='readout'),
                end_pin=Dict(component='coupling_pin', pin='open')
            ),
))
gui.rebuild()

11:16AM 51s WARNING [__init__]: Pin readout does not exist in component Qubit_1. cpw1 has not been built. Please check your pin_input values.


name:    cpw1
class:   RouteMeander          
options: 
  'chip'              : 'main',                       
  'layer'             : '1',                          
  'pin_inputs'        : {
       'start_pin'         : {
            'component'         : 'Qubit_1',                    
            'pin'               : 'readout',                    
                             },
       'end_pin'           : {
            'component'         : 'coupling_pin',               
            'pin'               : 'open',                       
                             },
                        },
  'fillet'            : '99.9um',                     
  'lead'              : {
       'start_straight'    : '100um',                      
       'end_straight'      : '0mm',                        
       'start_jogged_extension': '',                           
       'end_jogged_extension': '',                           
                        },
  'total_length'      : '4037um',        

In [7]:
# xmon_options = dict(
#     pos_x = '4.0mm',
#     pos_y = '3.55mm',
#     orientation = '90',
#     cross_width = '100um',
#     connection_pads=dict(
#         connector_1 = dict(connector_location = '90', connector_type = '0')
#     ),
# )

# Qubit_2 = TransmonCross(design, 'Qubit_2', options=xmon_options)


# gui.rebuild()
# gui.autoscale()
# gui.zoom_on_components(['Qubit_2'])

# LOM analysis

In [30]:
from qiskit_metal.analyses.quantization import LOManalysis
c1 = LOManalysis(design, "q3d")

In [31]:
c1.sim.setup

{'name': 'Setup',
 'reuse_selected_design': True,
 'reuse_setup': True,
 'freq_ghz': 5.0,
 'save_fields': False,
 'enabled': True,
 'max_passes': 15,
 'min_passes': 2,
 'min_converged_passes': 2,
 'percent_error': 0.5,
 'percent_refinement': 30,
 'auto_increase_solution_order': True,
 'solution_order': 'High',
 'solver_type': 'Iterative'}

In [32]:
c1.sim.setup.name = "Tune_Qubit_1"
c1.sim.setup.max_passes = 12
c1.sim.setup.min_converged_passes = 3
c1.sim.setup.percent_error = 0.05
# c1.sim.setup

In [33]:
c1.sim.run(name = "Tune_Qubit_1",
           components = ["Qubit_1"],
           open_terminations = [('Qubit_1','connector_1')])

INFO 05:46PM [connect_project]: Connecting to Ansys Desktop API...
INFO 05:46PM [load_ansys_project]: 	Opened Ansys App
INFO 05:46PM [load_ansys_project]: 	Opened Ansys Desktop v2025.1.0
INFO 05:46PM [load_ansys_project]: 	Opened Ansys Project
	Folder:    C:/Users/91566/Documents/Ansoft/
	Project:   Project18
INFO 05:46PM [connect_design]: 	Opened active design
	Design:    Tune_Qubit_1_q3d [Solution type: Q3D]
INFO 05:46PM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 05:46PM [connect]: 	Connected to project "Project18" and design "Tune_Qubit_1_q3d" 😀 

INFO 05:46PM [connect_design]: 	Opened active design
	Design:    Tune_Qubit_1_q3d [Solution type: Q3D]
INFO 05:46PM [get_setup]: 	Opened setup `Tune_Qubit_1`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 05:46PM [analyze]: Analyzing setup Tune_Qubit_1
INFO 05:47PM [get_matrix]: Exporting matrix data to (C:\Users\91566\AppData\Local\Temp\tmpia7ecfsq.txt, C, , Tune_Qubit_1:LastAdaptive, "Original", "ohm", "n

In [34]:
help(c1.sim)

Help on LumpedElementsSim in module qiskit_metal.analyses.simulation.lumped_elements object:

class LumpedElementsSim(qiskit_metal.analyses.core.simulation.QSimulation)
 |  LumpedElementsSim(design: 'QDesign' = None, renderer_name: str = 'q3d')
 |  
 |  Compute Capacitance matrix using the selected renderer.
 |  
 |  Default Setup:
 |      * name (str): Name of capacitive simulation setup. Defaults to "Setup".
 |      * freq_ghz (int): Frequency in GHz. Defaults to 5.
 |      * save_fields (bool): Whether or not to save fields. Defaults to False.
 |      * enabled (bool): Whether or not setup is enabled. Defaults to True.
 |      * max_passes (int): Maximum number of passes. Defaults to 15.
 |      * min_passes (int): Minimum number of passes. Defaults to 2.
 |      * min_converged_passes (int): Minimum number of converged passes.
 |          Defaults to 2.
 |      * percent_error (float): Error tolerance as a percentage. Defaults to 0.5.
 |      * percent_refinement (int): Refinement 

In [35]:
c1.sim.capacitance_matrix

,connector_1_connector_arm_Qubit_1,cross_Qubit_1,ground_main_plane
connector_1_connector_arm_Qubit_1,38.14685,-3.53094,-34.43668
cross_Qubit_1,-3.53094,136.36750,-130.21965
ground_main_plane,-34.43668,-130.21965,205.26329


# EPR analysis

In [ ]:
import os
import sys

target_path = os.path.abspath('C:/Users/91566/Desktop/rz/qubit_design/automation')
if target_path not in sys.path:
    sys.path.append(target_path)
from epranalysis import epr_components

comp_list = ["Qubit_1","cpw1"]

epr_options = dict(
    max_passes = 12,
    vars = dict(
        Lj = "10nH",
    ),
    setup_update = dict(
        max_delta_f = 0.1,
        min_freq_ghz = 1.1,
        n_modes = 2,
    ),
    sim_run_options = dict(
        open_terminations = [],
        port_list = []
    )
)

results = epr_components(comp_list=comp_list, design=design, options=epr_options)

INFO 09:15PM [connect_project]: Connecting to Ansys Desktop API...
INFO 09:15PM [load_ansys_project]: 	Opened Ansys App
INFO 09:15PM [load_ansys_project]: 	Opened Ansys Desktop v2025.1.0
INFO 09:15PM [load_ansys_project]: 	Opened Ansys Project
	Folder:    C:/Users/91566/Documents/Ansoft/
	Project:   Project26
INFO 09:15PM [connect_design]: 	Opened active design
	Design:    sim_hfss [Solution type: Eigenmode]
INFO 09:15PM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)
INFO 09:15PM [connect]: 	Connected to project "Project26" and design "sim_hfss" 😀 

INFO 09:15PM [connect_design]: 	Opened active design
	Design:    sim_hfss [Solution type: Eigenmode]
09:15PM 23s ERROR [render_element_path]: We cannot find a writable design. 
  Either you are trying to use a Ansys design that is not empty, in which case please clear it manually or with the renderer method clean_active_design(). 
  Or you accidentally deleted the design in Ansys, in which case please create a new o

com_error:  (-2147352567, 'Exception occurred.', (0, None, None, None, 0, -2147024381), None)


com_error: (-2147352567, 'Exception occurred.', (0, None, None, None, 0, -2147024381), None)

In [46]:
print(results)

HamiltonianResultsContainer([('0', OrderedDict([('f_0', 0     4151.810116
1    28135.404490
2    28208.436281
Name: 0, dtype: float64), ('f_1', 0     4025.407816
1    28135.401553
2    28208.434469
dtype: float64), ('f_ND', 0     4021.353822+    0.000000j
1    28135.401567+    0.000000j
2    28208.434478+    0.000000j
dtype: complex128), ('chi_O1',             0             1             2
0  126.397552  5.873734e-03  3.623305e-03
1    0.005874  6.823858e-08  8.418807e-08
2    0.003623  8.418807e-08  2.596636e-08), ('chi_ND',                         0                                               1  \
0  135.212705+  0.000000j  5.821651e-03+0.000000e+                    00j   
1    0.005822+  0.000000j  6.667328e-08+0.000000e+                    00j   
2    0.003590+  0.000000j  8.279419e-08+0.000000e+                    00j   

                                                2  
0  3.590424e-03+0.000000e+                    00j  
1  8.279419e-08+0.000000e+                    00j  
2  

In [48]:
from qiskit_metal.analyses.quantization import EPRanalysis

In [49]:
eig_qb = EPRanalysis(design, "hfss")

In [50]:
eig_qb.sim.setup

{'name': 'Setup',
 'reuse_selected_design': True,
 'reuse_setup': True,
 'min_freq_ghz': 1,
 'n_modes': 1,
 'max_delta_f': 0.5,
 'max_passes': 10,
 'min_passes': 1,
 'min_converged': 1,
 'pct_refinement': 30,
 'basis_order': 1,
 'vars': {'Lj': '10 nH', 'Cj': '0 fF'}}

In [51]:
eig_qb.sim.setup.max_passes = 12
eig_qb.sim.setup.vars.Lj = '10 nH'

eig_qb.sim.setup_update(max_delta_f = 0.1, 
                        min_freq_ghz = 1.1,
                        n_modes=2)

eig_qb.sim.setup

{'name': 'Setup',
 'reuse_selected_design': True,
 'reuse_setup': True,
 'min_freq_ghz': 1.1,
 'n_modes': 2,
 'max_delta_f': 0.1,
 'max_passes': 12,
 'min_passes': 1,
 'min_converged': 1,
 'pct_refinement': 30,
 'basis_order': 1,
 'vars': {'Lj': '10 nH', 'Cj': '0 fF'}}

In [57]:
eig_qb.sim.run(name="Qbit", components=['Qubit_1','cpw1'], open_terminations=[],port_list=[])
eig_qb.sim.plot_convergences()

INFO 04:26PM [connect_design]: 	Opened active design
	Design:    Qbit_hfss [Solution type: Eigenmode]
04:26PM 51s ERROR [render_element_path]: We cannot find a writable design. 
  Either you are trying to use a Ansys design that is not empty, in which case please clear it manually or with the renderer method clean_active_design(). 
  Or you accidentally deleted the design in Ansys, in which case please create a new one.


com_error:  (-2147352567, 'Exception occurred.', (0, None, None, None, 0, -2147024381), None)


com_error: (-2147352567, 'Exception occurred.', (0, None, None, None, 0, -2147024381), None)

In [ ]:
eig_qb.sim.plot_fields('main')
eig_qb.sim

INFO 04:25PM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)


In [ ]:
eig_qb.setup

{'junctions': {'jj': {'Lj_variable': 'Lj',
   'Cj_variable': 'Cj',
   'rect': '',
   'line': ''}},
 'dissipatives': {'dielectrics_bulk': ['main']},
 'cos_trunc': 8,
 'fock_trunc': 7,
 'sweep_variable': 'Lj'}

In [ ]:
eig_qb.setup.junctions.jj.rect = 'JJ_rect_Lj_Qubit_1_rect_jj'
eig_qb.setup.junctions.jj.line = 'JJ_Lj_Qubit_1_rect_jj_'
# eig_qb.setup.junctions.jj.rect = 'JJ_rect_Lj_Qubit_2_rect_jj'
# eig_qb.setup.junctions.jj.line = 'JJ_Lj_Qubit_2_rect_jj_'
eig_qb.setup

{'junctions': {'jj': {'Lj_variable': 'Lj',
   'Cj_variable': 'Cj',
   'rect': 'JJ_rect_Lj_Qubit_1_rect_jj',
   'line': 'JJ_Lj_Qubit_1_rect_jj_'}},
 'dissipatives': {'dielectrics_bulk': ['main']},
 'cos_trunc': 8,
 'fock_trunc': 7,
 'sweep_variable': 'Lj'}

In [ ]:
eig_qb.run_epr()
print(eig_qb.sim.renderer.epr_quantum_analysis.results['0']['chi_O1'])

Design "Qbit_hfss" info:
	# eigenmodes    2
	# variations    1
Design "Qbit_hfss" info:
	# eigenmodes    2
	# variations    1

        energy_elec_all       = 7.97828154676379e-26
        energy_elec_substrate = 7.34103037473868e-26
        EPR of substrate = 92.0%

        energy_mag    = 2.41562224658763e-28
        energy_mag % of energy_elec_all  = 0.3%
        

Variation 0  [1/1]

  Mode 0 at 4.19 GHz   [1/2]
    Calculating ℰ_magnetic,ℰ_electric
       (ℰ_E-ℰ_H)/ℰ_E       ℰ_E       ℰ_H
               99.7%  3.989e-26 1.208e-28

    Calculating junction energy participation ration (EPR)
	method=`line_voltage`. First estimates:
	junction        EPR p_0j   sign s_0j    (p_capacitive)
		Energy fraction (Lj over Lj&Cj)= 98.63%
	jj              0.996071  (+)        0.0137966
		(U_tot_cap-U_tot_ind)/mean=0.73%
Calculating Qdielectric_main for mode 0 (0/1)
p_dielectric_main_0 = 0.9201267631018127

  Mode 1 at 28.18 GHz   [2/2]
    Calculating ℰ_magnetic,ℰ_electric
       (ℰ_E-ℰ_H)/ℰ_E  

WARNING 04:25PM [__init__]: <p>Error: <class 'IndexError'></p>
ERROR 04:25PM [_get_participation_normalized]: WARNING: U_tot_cap-U_tot_ind / mean = 116.5% is > 15%.                     
Is the simulation converged? Proceed with caution
ERROR 04:25PM [_get_participation_normalized]: WARNING: U_tot_cap-U_tot_ind / mean = 116.5% is > 15%.                     
Is the simulation converged? Proceed with caution



ANALYSIS DONE. Data saved to:

C:\data-pyEPR\Project23\Qbit_hfss\2025-07-25 16-25-53.npz


	 Differences in variations:



 . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 
Variation 0

Starting the diagonalization
Finished the diagonalization
sdag: Quantum object: dims=[[1, 1], [7, 7]], shape=(1, 49), type='bra', dtype=Dense
Qobj data =
[[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0.]]
sdag: Quantum object: dims=[[1, 1], [7, 7]], shape=(1, 49), type='bra', dtype=Dense
Qobj data =
[[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0.]]
sdag: Quantum object: dims=[[1, 1], [7, 7]], shape=(1, 49), type='bra', dtype=Dense
Qobj data =
[[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0

#### Mode frequencies (MHz)

###### Numerical diagonalization

Lj,10
0,4050.46+ 0.00j
1,28183.78+ 0.00j


#### Kerr Non-linear coefficient table (MHz)

###### Numerical diagonalization

0                               1
Lj                                                                  
10 0  1.43e+02+0.00e+            00j  6.00e-03+0.00e+            00j
   1  6.00e-03+0.00e+            00j  6.78e-08+0.00e+            00j

            0             1
0  133.343214  6.048045e-03
1    0.006048  6.858026e-08
